# Projet EcoTrack Assistant - Notebook de départ

Complétez uniquement les cellules marquées **TODO**. Le socle reprend le TP : chargement, découpage, embeddings, similarité cosinus, top-3, génération Gemini, citations, refus hors corpus et précision@3.


In [ ]:
!pip -q install sentence-transformers google-genai pandas numpy


In [ ]:
import os, glob, re, json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from google import genai


## 1. Charger le corpus
Téléversez le dossier `corpus_ecotrack` dans Colab ou montez Google Drive.


In [ ]:
CORPUS_DIR = "/content/corpus_ecotrack"

# TODO 1 : charger tous les fichiers .txt dans un dictionnaire {nom_fichier: texte}
documents = {}
for path in glob.glob(os.path.join(CORPUS_DIR, "*.txt")):
    # À COMPLÉTER
    pass

print(f"Documents chargés : {len(documents)}")


## 2. Découper les documents en passages
Conseil : passages de 400 à 700 caractères, avec un petit chevauchement.


In [ ]:
def decouper_texte(texte, taille=600, chevauchement=100):
    # TODO 2 : retourner une liste de passages non vides
    passages = []
    return passages

chunks = []
for source, texte in documents.items():
    for i, passage in enumerate(decouper_texte(texte)):
        chunks.append({"chunk_id": f"{source}#P{i+1}", "source": source, "texte": passage})

print(f"Passages créés : {len(chunks)}")


## 3. Calculer les embeddings


In [ ]:
modele_embeddings = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# TODO 3 : encoder les textes des passages avec normalize_embeddings=True
embeddings_chunks = None


## 4. Recherche sémantique par similarité cosinus


In [ ]:
def rechercher(question, k=3):
    # TODO 4 : encoder la question, calculer les similarités et retourner les k meilleurs passages
    # Astuce : avec des vecteurs normalisés, le produit matriciel correspond à la similarité cosinus.
    resultats = []
    return resultats


## 5. Génération avec Gemini
Placez la clé dans les secrets Colab. Ne l'écrivez jamais en clair dans le notebook.


In [ ]:
from google.colab import userdata
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL_NAME = "gemini-2.5-flash"


In [ ]:
SEUIL_CONFIANCE = 0.35  # à justifier ou ajuster

def construire_contexte(resultats):
    return "\n\n".join([f"[{r['chunk_id']}] {r['texte']}" for r in resultats])

def repondre(question):
    resultats = rechercher(question, k=3)
    # TODO 5 : si aucun score n'est suffisant, retourner un refus poli
    # TODO 6 : construire un prompt imposant une réponse fondée uniquement sur le contexte
    # TODO 7 : demander des citations au format [nom_fichier#Pxx]
    contexte = construire_contexte(resultats)
    prompt = f"""À COMPLÉTER
QUESTION : {question}
CONTEXTE :
{contexte}
"""
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    return response.text, resultats


## 6. Essai rapide


In [ ]:
question = "À partir de quel niveau un conteneur est-il critique ?"
# TODO 8 : appeler repondre et afficher la réponse puis les trois passages récupérés


## 7. Jeu de test et précision@3
La précision@3 demandée ici mesure la proportion de questions pour lesquelles le document attendu apparaît parmi les trois passages retournés.


In [ ]:
tests = pd.read_csv("/content/jeu_test_10_questions.csv", sep=";")

def precision_at_3(tests):
    succes = 0
    details = []
    for _, ligne in tests.iterrows():
        # TODO 9 : chercher le top-3 et vérifier la présence du document attendu
        trouve = False
        details.append({"id": ligne["id"], "trouve": trouve})
        succes += int(trouve)
    return succes / len(tests), pd.DataFrame(details)

score, details = precision_at_3(tests)
print(f"Précision@3 : {score:.1%}")
display(details)


## 8. Analyse d'une erreur
Choisissez une question ratée. Expliquez :
1. ce qui a été récupéré ;
2. pourquoi le document attendu n'est pas dans le top-3 ;
3. une amélioration possible : découpage, formulation, seuil, corpus ou modèle d'embeddings.


In [ ]:
# TODO 10 : votre analyse d'erreur ici


## BONUS A - Tool Use : `etat_conteneurs(zone)` (+2)


In [ ]:
conteneurs = pd.read_csv("/content/etat_conteneurs.csv", sep=";")

def etat_conteneurs(zone):
    # TODO BONUS : filtrer la zone et retourner un résumé lisible
    pass


## BONUS B - Reflection (+1)
Ajoutez une seconde étape qui critique la réponse selon trois critères : fidélité au contexte, citations et absence d'invention. Comparez avant/après sur les 10 questions.


In [ ]:
# TODO BONUS : fonction reflection(reponse, contexte)


## BONUS C - Injection de prompt documentaire (+1)
Ajoutez `bonus_document_injection.txt` au corpus. Vérifiez que l'assistant n'obéit pas à l'instruction malveillante et expliquez la protection mise en place.


In [ ]:
# TODO BONUS : test et résultat attendu
